Missouri Manual Semantic Categories:
•Agriculture
•Cities
•Community engagement
•Cost of living – Services – Healthcare
•Culture
•Diversity
•Economy/Commerce/Industry
•Environment
•Ideology
•Infrastructure
•Elderly
•Environment
•Family – Children
•K12
•Named neighborhood
•NIMBY
•Policing
•Poverty
•Recreation – Tourism
•Religion
•Suburbs
•Technology
•University
•Violence
•Vulnerable populations

### Imports

In [ ]:
# pip install transformers spacy torch torchvision
# python -m spacy download en_core_web_sm

import pandas as pd
import spacy
from transformers import pipeline
import json

### Init

##### Community Labels

In [ ]:
candidate_labels = [
    "Agriculture", "Cities", "Community engagement", "Cost of living", 
    "Culture", "Diversity", "Economy and Commerce", "Environment", 
    "Ideology", "Infrastructure", "Elderly", "Family and Children", 
    "K-12 Education", "Named neighborhood", "NIMBY", "Policing", 
    "Poverty", "Recreation and Tourism", "Religion", "Suburbs", 
    "Technology", "University", "Violence", "Vulnerable populations"
]

#### Load Models

In [ ]:
nlp_spacy = spacy.load("en_core_web_sm")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

#### Functions

In [ ]:
def remove_geography(text):
    if not isinstance(text, str):
        return ""
    doc = nlp_spacy(text)
    # remove geopolitical entities and locations
    # why? mggg models were overfitting to geography
    clean_text = " ".join([token.text for token in doc if token.ent_type_ not in ['GPE', 'LOC']])
    return clean_text

def categorize_comment(text):
    if len(text.strip()) < 5:
        return "Unknown"
    result = classifier(text, candidate_labels, multi_label=False)
    
    # top probability label
    return result['labels'][0]


### Run Categorize

In [ ]:
# open as json
with open("data/Colorado_communities.geojson", "r") as f:
    data = json.load(f)
features = data['features']
df_communities = pd.DataFrame([f['properties'] for f in features])


def prepare_text(row):
    # combine testimony fields
    fields = ['cultural_interests', 'economic_interests', 'comm_activities', 'other_considerations']
    combined = " ".join([str(row.get(f, "")) for f in fields])
    return " ".join(combined.split()) # Clean whitespace

# get all testimonies, then categorize
df_communities['clean_text'] = df_communities.apply(prepare_text, axis=1)
df_communities['predicted_category'] = df_communities['clean_text'].apply(categorize_comment)

# back to geojson for plotting
for i, feature in enumerate(data['features']):
    feature['properties']['predicted_category'] = df_communities.iloc[i]['predicted_category']

with open("Colorado_communities_labeled.geojson", "w") as f:
    json.dump(data, f)

#### Debug

In [23]:
df_communities['predicted_category'].value_counts()

predicted_category
Community engagement      112
Named neighborhood         55
Diversity                  37
Vulnerable populations     27
Cost of living             16
Environment                16
Unknown                    14
University                 10
Family and Children        10
Culture                     8
Elderly                     6
Cities                      5
Religion                    5
Recreation and Tourism      4
NIMBY                       3
Infrastructure              2
Economy and Commerce        2
Suburbs                     2
Poverty                     1
Agriculture                 1
Violence                    1
Policing                    1
K-12 Education              1
Name: count, dtype: int64

In [22]:
df_communities[['entry_name', 'clean_text','predicted_category']].head(20)

,entry_name,clean_text,predicted_category
0,Montbello,My community is largely Hispanic. There are fo...,Community engagement
1,Platt Park,Platt Park is a community of involved citizens...,Community engagement
2,Curtis Park,My community is Curtis Park which is a designa...,Named neighborhood
3,Gabby in Boulder,My community is made up of the University of C...,University
4,Northglenn,My community is made of Latinos and Hispanics,Diversity
5,Buffalo Highland,My community buffalo highlands needs a park. W...,Community engagement
6,GES,My community is primarily comprised of generat...,Community engagement
7,University of Colorado Colorado Springs Geography,The university community includes both the cam...,University
8,Scared Chickens,If map drawers were to draw our community they...,Vulnerable populations
9,Mayfair,Very white and seemingly upper middle class. W...,NIMBY
